In [1]:
import os
from pathlib import Path
import json
from jsonargparse import CLI
import boto3

import time
from copy import deepcopy
import threading
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TextGenerationPipeline
from peft import PeftModel
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

def get_batch_response(text, base_model, model_path, temperature, max_tokens, EXSTING):
    """
    Modified get_response function to use local model instead of API calls.
    model_path: path to the local model directory
    tokenizer and model: optional pre-loaded tokenizer and model objects
    """
    
    while True:
        try:
            # content = prompt.format(text)
            
            # Load model and tokenizer if not provided (for thread safety)
            
            # device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

            model = LLM(model=base_model, 
                        enable_lora=True,
                        tensor_parallel_size=8,
                        dtype="bfloat16")

            lora = LoRARequest(lora_name="llama-3.2-instruct",
                                lora_int_id=1,
                                lora_local_path=model_path)

            sampling_params = SamplingParams(max_tokens=max_tokens,
                                            temperature=temperature,
                                            top_p=0.9)

            outputs = model.generate(text, sampling_params, lora_request=lora)

            results = [output.outputs[0].text for output in outputs]
            
            ##############################################
            ################# Approach 1 #################
            ##############################################

            # # Load tokenizer
            # tokenizer = AutoTokenizer.from_pretrained(base_model)
            # base_model = AutoModelForCausalLM.from_pretrained(base_model,  
            #                     torch_dtype=torch.float16,
            #                     device_map='auto'
            #                     )

            # # Load LoRA adapter on top of base model
            # model = PeftModel.from_pretrained(base_model, model_path)

            # if tokenizer.pad_token is None:
            #     tokenizer.pad_token = tokenizer.eos_token
            
            # tokenizer.padding_side = "left"
            
            # model = model.merge_and_unload()

            # model.eval()
            # results=[]

            # pipe = TextGenerationPipeline(model=model, tokenizer=tokenizer)

            # results = pipe(text, max_new_tokens=max_tokens,
            #                 temperature=temperature,
            #                 do_sample=temperature > 0,
            #                 pad_token_id=tokenizer.pad_token_id,
            #                 eos_token_id=tokenizer.eos_token_id,
            #                 use_cache=True,
            #                 batch_size=len(text)
            #             )

            # results = [result[0]['generated_text'][len(prompt):].strip() if result[0]['generated_text'].startswith(prompt) else result[0]['generated_text']
            #                     for prompt, result in zip(text, results)]
            

            ##############################################
            ################# Approach 2 #################
            ##############################################

            # for content in text:
            #     # Tokenize input
            #     inputs = tokenizer(
            #         content,
            #         return_tensors="pt",
            #         truncation=True,
            #         max_length=512,
            #         padding=True
            #     )
            
            #     # Move to device
            #     device = next(model.parameters()).device
            #     inputs = {k: v.to(device) for k, v in inputs.items()}
            
            #     # Generate response
            #     with torch.no_grad():
            #         outputs = model.generate(
            #             **inputs,
            #             max_new_tokens=max_tokens,
            #             temperature=temperature,
            #             do_sample=temperature > 0,
            #             pad_token_id=tokenizer.pad_token_id,
            #             eos_token_id=tokenizer.eos_token_id,
            #             use_cache=True
            #         )
            
            #     # Decode response (only the new tokens)
            #     input_length = inputs['input_ids'].shape[1]
            #     generated_tokens = outputs[0][input_length:]
            #     response_text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
            #     results.append(response_text)
            
            
            return results
            
            
        except Exception as e:
            if e == KeyboardInterrupt:
                raise e
            print(f"Error: {e}")
            time.sleep(2)
            continue

        break


def main(from_json: str = None, to_json: str = None, prompt: str = None, base_model: str = 'llama-3.1-instruct',
         model_path: str = 'llama-3.1-instruct', temperature: float = 0, max_tokens: int = 512, 
         batch_size: int = 8, n_print: int = 100, n_samples: int = -1, 
         input_field: str = 'input', existing_json: str = None):
    EXSTING = {}
    if existing_json is not None:
        with open(existing_json, 'r') as f:
            for l in f.readlines():
                d = json.loads(l)
                if d['resp'] != 'API Failed':
                    EXSTING[d['prompt']] = d
    
    
    path = Path(to_json)
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        path.touch()

    with open(from_json, "r") as fr, open(to_json, 'w') as fw:

        results = []

        lines = fr.readlines()
        total_lines = min(len(lines), n_samples) if n_samples > 0 else len(lines)
        start_time = time.time()
        
        for i in range(0, len(lines), batch_size):
           
            batch = [prompt.format(json.loads(lines[i+j])[input_field]) for j in range(min(batch_size, len(lines)-i))]
            
            batch_results = get_batch_response(
                                batch, base_model, model_path, temperature, max_tokens, EXSTING
                            )
            
            for result in batch_results:
                fw.write(json.dumps(result) + '\n')

            if i % n_print == 0:
                print(f'Time elapsed: {time.time() - start_time:.2f} sec. {i+8} / {total_lines} samples generated. ')

main(from_json='testsets/inspired/test_clean.jsonl',
    to_json='test_res/inspired/llama-3.2-instruct/inspired_test_clean.jsonl',
    prompt="Pretend you are a movie recommender system. I will give you a conversation between a user and you (a recommender system). Based on the conversation, you reply with a list of 20 recommendations in the format of '1. [Movie Name]\n 2. [Movie Name]\n ...' with no extra sentences no user reponse. Here is the conversation: {}",
    base_model='meta-llama/Llama-3.2-1B-Instruct',
    model_path='../outputs/sft/inspired/test',
    temperature=0.1,
    max_tokens=512,
    n_print=1,
    n_samples=-1)

/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 06-27 05:06:36 [__init__.py:244] Automatically detected platform cuda.


2025-06-27 05:06:38,410	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 06-27 05:06:45 [config.py:823] This model supports multiple tasks: {'embed', 'generate', 'classify', 'reward', 'score'}. Defaulting to 'generate'.
INFO 06-27 05:06:45 [config.py:1946] Defaulting to use mp for distributed inference
INFO 06-27 05:06:45 [config.py:2195] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 06-27 05:06:47 [core.py:455] Waiting for init message from front-end.
INFO 06-27 05:06:47 [core.py:70] Initializing a V1 LLM engine (v0.9.1) with config: model='meta-llama/Llama-3.2-1B-Instruct', speculative_config=None, tokenizer='meta-llama/Llama-3.2-1B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=131072, download_dir=None, load_format=auto, tensor_parallel_size=8, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_c

Process EngineCore_0:
Traceback (most recent call last):
  File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 519, in run_engine_core
    raise e
  File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 506, in run_engine_core
    engine_core = EngineCoreProc(*args, **kwargs)
  File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 390, in __init__
    super().__init__(vllm_config, executor_class, log_stats,
  File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 76, in __init__
    sel

...

/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/multiprocessing/resource_tracker.py:224: UserWarning: resource_tracker: There appear to be 8 leaked shared_memory objects to clean up at shutdown
  warnings.warn('resource_tracker: There appear to be %d '


INFO 06-27 05:07:06 [config.py:823] This model supports multiple tasks: {'embed', 'generate', 'classify', 'reward', 'score'}. Defaulting to 'generate'.
INFO 06-27 05:07:06 [config.py:1946] Defaulting to use mp for distributed inference
INFO 06-27 05:07:06 [config.py:2195] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 06-27 05:07:06 [core.py:455] Waiting for init message from front-end.
INFO 06-27 05:07:06 [core.py:70] Initializing a V1 LLM engine (v0.9.1) with config: model='meta-llama/Llama-3.2-1B-Instruct', speculative_config=None, tokenizer='meta-llama/Llama-3.2-1B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=131072, download_dir=None, load_format=auto, tensor_parallel_size=8, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_c

(VllmWorker rank=5 pid=75041) Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7f534461d750>>
(VllmWorker rank=5 pid=75041) Traceback (most recent call last):
(VllmWorker rank=5 pid=75041)   File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
(VllmWorker rank=5 pid=75041)     def _clean_thread_parent_frames(
(VllmWorker rank=5 pid=75041)   File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/site-packages/vllm/v1/executor/multiproc_executor.py", line 455, in signal_handler
(VllmWorker rank=5 pid=75041)     raise SystemExit()
(VllmWorker rank=5 pid=75041) SystemExit: 


ERROR 06-27 05:07:18 [core.py:515] EngineCore failed to start.
ERROR 06-27 05:07:18 [core.py:515] Traceback (most recent call last):
ERROR 06-27 05:07:18 [core.py:515]   File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 506, in run_engine_core
ERROR 06-27 05:07:18 [core.py:515]     engine_core = EngineCoreProc(*args, **kwargs)
ERROR 06-27 05:07:18 [core.py:515]   File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 390, in __init__
ERROR 06-27 05:07:18 [core.py:515]     super().__init__(vllm_config, executor_class, log_stats,
ERROR 06-27 05:07:18 [core.py:515]   File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 76, in __init__
ERROR 06-27 05:07:18 [core.py:515]     self.model_executor = executor_class(vllm_config)
ERROR 06-27 05:07:18 [core.py:515]   File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/site

Process EngineCore_0:
Traceback (most recent call last):
  File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 519, in run_engine_core
    raise e
  File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 506, in run_engine_core
    engine_core = EngineCoreProc(*args, **kwargs)
  File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 390, in __init__
    super().__init__(vllm_config, executor_class, log_stats,
  File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 76, in __init__
    sel